# Model Training

In [ ]:
"""
Two-phase fine-tuning pipeline for game action recognition (10-class UCF101 subset).
Rebuilt to match Chapter 3 (Methodology) EXACTLY - see inline comments citing the
relevant Chapter 3 section for every design choice.

Folder structure expected (already split 70:15:15, stratified by class):
dataset/
    train/<class_name>/*.jpg
    val/<class_name>/*.jpg
    test/<class_name>/*.jpg

Everything needed for Chapter 4 gets written to results/<model_name>/:
    - training_log_phase1.csv, training_log_phase2.csv   (per-epoch acc/loss -> training curves, Section 3.5)
    - classification_report.csv                          (accuracy, macro precision/recall/F1, Section 3.5)
    - confusion_matrix.csv + confusion_matrix.png         (10x10, Section 3.5)
    - best_model.keras
And a combined results/model_comparison_summary.csv across all 4 models.

ASSUMPTIONS MADE where Chapter 3 does not give an exact number (flagged so you can
adjust or add the missing number into Chapter 3 for consistency):
    - Phase 1 learning rate for the 3 transfer-learning models: not stated in 3.4.3
      (only the Phase 2 rate of 1e-5 is given). Set to 0.001, matching the Custom
      CNN's stated rate and standard practice for frozen-base training.
    - Phase 2 max epoch cap for transfer-learning models: 3.4.3 says early stopping
      on val_loss governs Phase 2 but gives no epoch ceiling. Set to 30 (generous
      ceiling; early stopping will normally end it sooner).
    - Early stopping patience in Phase 2 for transfer models: not stated. Set to 10,
      matching the Custom CNN's stated patience (Section 3.4.2).
"""


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# =========================================================================
# CONFIG - matches Chapter 3, Sections 3.3 and 3.4
# =========================================================================
DATA_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset"  # my actual dataset location
IMG_SIZE = (224, 224)          # Section 3.3: standard size for VGG16/ResNet50/EfficientNet-B0,
BATCH_SIZE = 32                #   and the custom CNN uses the same resolution for comparability
NUM_CLASSES = 10

# --- QUICK TEST TOGGLE ---
# Set to True first, to confirm the whole pipeline runs end-to-end with tiny
# epoch counts (a few minutes total). Once that works with no errors, set to
# False and re-run for the real Chapter 4 numbers (this will take much longer).
QUICK_TEST = True

if QUICK_TEST:
    CUSTOM_CNN_MAX_EPOCHS = 2
    PHASE1_EPOCHS = 2
    PHASE2_MAX_EPOCHS = 2
else:
    CUSTOM_CNN_MAX_EPOCHS = 50     # Section 3.4.2
    PHASE1_EPOCHS = 20             # Section 3.4.3 - fixed, no early stopping in Phase 1
    PHASE2_MAX_EPOCHS = 30         # ASSUMPTION - Ch3 gives no ceiling, only early stopping

CUSTOM_CNN_LR = 0.001          # Section 3.4.2
CUSTOM_CNN_PATIENCE = 10       # Section 3.4.2 - early stopping on val_loss

PHASE1_LR = 0.001              # ASSUMPTION (see module docstring) - not stated for transfer models
PHASE2_LR = 1e-5               # Section 3.4.3 - explicit
PHASE2_PATIENCE = 10           # ASSUMPTION, mirrors Custom CNN's patience
UNFREEZE_FRACTION = 0.30       # Section 3.4.3 - "top 30% of layers" unfrozen in Phase 2

RESULTS_DIR = "/content/drive/MyDrive/ucf101_pipeline/results"  # saved to Drive, survives disconnects
SEED = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
tf.random.set_seed(SEED)

In [ ]:
# =========================================================================
# DATA PIPELINE - Section 3.3
# Uses Keras's ImageDataGenerator (named explicitly in Ch3) rather than the
# newer tf.data preprocessing-layers API, to match what Chapter 3 states.
# =========================================================================
# Augmentation parameters exactly as listed in Section 3.3:
#   horizontal flip p=0.5, rotation +/-15deg, width/height shift +/-10%,
#   zoom +/-10%, brightness +/-20%. Applied to the TRAINING set only.
train_datagen_custom = ImageDataGenerator(
    rescale=1.0 / 255,                       # Section 3.3: pixel values normalised to [0,1]
    horizontal_flip=True,                    # p=0.5 is ImageDataGenerator's default flip behaviour
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    brightness_range=(0.8, 1.2),              # +/-20%
)
eval_datagen_custom = ImageDataGenerator(rescale=1.0 / 255)  # val/test: NO augmentation (Section 3.3)


def make_transfer_datagens(preprocess_fn):
    """For the 3 transfer models: same augmentation, but pixel handling goes through
    the model-specific Keras Applications preprocessing function (ImageNet channel-wise
    mean subtraction) instead of a plain /255 rescale - Section 3.3, second paragraph."""
    train_dg = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        horizontal_flip=True,
        rotation_range=15,
        width_shift_range=0.10,
        height_shift_range=0.10,
        zoom_range=0.10,
        brightness_range=(0.8, 1.2),
    )
    eval_dg = ImageDataGenerator(preprocessing_function=preprocess_fn)
    return train_dg, eval_dg


def flow(datagen, split, shuffle):
    return datagen.flow_from_directory(
        os.path.join(DATA_DIR, split),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=shuffle,
        seed=SEED,
    )

In [ ]:
# =========================================================================
# CUSTOM CNN BASELINE - Section 3.4.2, exact architecture
# =========================================================================
def build_custom_cnn():
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, padding="same")(x)   # 3x3 filters
        x = layers.BatchNormalization()(x)                 # BatchNorm after each conv layer
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)                       # 2x2 max-pooling
        return x

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = conv_block(inputs, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256)                                  # filters: 32, 64, 128, 256
    x = layers.GlobalAveragePooling2D()(x)                  # GAP, not Flatten (Section 3.4.2)
    x = layers.Dense(512, activation="relu")(x)             # FC layer, 512 units
    x = layers.Dropout(0.5)(x)                              # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

In [ ]:
# =========================================================================
# TRANSFER LEARNING MODELS - Section 3.4.3, exact head architecture
# =========================================================================
def build_transfer_model(base_class):
    base = base_class(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False  # Phase 1: base fully frozen

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)   # FC layer, 256 units (Section 3.4.3)
    x = layers.Dropout(0.5)(x)                    # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    return model, base


TRANSFER_SPECS = {
    "VGG16": (VGG16, tf.keras.applications.vgg16.preprocess_input),
    "ResNet50": (ResNet50, tf.keras.applications.resnet50.preprocess_input),
    "EfficientNetB0": (EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
}

In [ ]:
# =========================================================================
# TRAIN + EVALUATE: CUSTOM CNN (single phase, Section 3.4.2)
# =========================================================================
def train_custom_cnn(class_names):
    name = "Custom_CNN"
    print(f"\n{'='*70}\nTraining: {name}\n{'='*70}")
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    train_gen = flow(train_datagen_custom, "train", shuffle=True)
    val_gen = flow(eval_datagen_custom, "val", shuffle=True)
    test_gen = flow(eval_datagen_custom, "test", shuffle=False)

    model = build_custom_cnn()
    model.compile(
        optimizer=optimizers.Adam(learning_rate=CUSTOM_CNN_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    ckpt_path = os.path.join(out_dir, "best_model.keras")
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=CUSTOM_CNN_PATIENCE, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase1.csv")),
    ]
    model.fit(train_gen, validation_data=val_gen, epochs=CUSTOM_CNN_MAX_EPOCHS, callbacks=callbacks)

    model = tf.keras.models.load_model(ckpt_path)
    return evaluate_and_save(model, test_gen, class_names, out_dir, name)

In [ ]:
# =========================================================================
# TRAIN + EVALUATE: ONE TRANSFER MODEL (two phases, Section 3.4.3)
# =========================================================================
def train_transfer_model(name, base_class, preprocess_fn, class_names):
    print(f"\n{'='*70}\nTraining: {name}\n{'='*70}")
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    train_dg, eval_dg = make_transfer_datagens(preprocess_fn)
    train_gen = flow(train_dg, "train", shuffle=True)
    val_gen = flow(eval_dg, "val", shuffle=True)
    test_gen = flow(eval_dg, "test", shuffle=False)

    model, base = build_transfer_model(base_class)
    ckpt_path = os.path.join(out_dir, "best_model.keras")

    # ---------------- PHASE 1: frozen base, head only, 20 fixed epochs, NO early stopping ----------------
    model.compile(optimizer=optimizers.Adam(learning_rate=PHASE1_LR),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        train_gen, validation_data=val_gen, epochs=PHASE1_EPOCHS,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
            tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase1.csv")),
        ],
    )

    # ---------------- PHASE 2: unfreeze top 30% of base, LR=1e-5, early stop on val_loss ----------------
    base.trainable = True
    freeze_until = int(len(base.layers) * (1 - UNFREEZE_FRACTION))
    for layer in base.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(learning_rate=PHASE2_LR),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        train_gen, validation_data=val_gen, epochs=PHASE2_MAX_EPOCHS,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PHASE2_PATIENCE, restore_best_weights=True),
            tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase2.csv")),
        ],
    )

    model = tf.keras.models.load_model(ckpt_path)
    return evaluate_and_save(model, test_gen, class_names, out_dir, name)

In [ ]:

# =========================================================================
# SHARED EVALUATION - Section 3.5 (accuracy, macro P/R/F1, confusion matrix)
# =========================================================================
def evaluate_and_save(model, test_gen, class_names, out_dir, name):
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes  # ground-truth labels in the (unshuffled) test generator order

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(os.path.join(out_dir, "classification_report.csv"))

    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(os.path.join(out_dir, "confusion_matrix.csv"))

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.close()

    summary_row = {
        "Model": name,
        "Top1_Accuracy": report["accuracy"],
        "Precision_Macro": report["macro avg"]["precision"],
        "Recall_Macro": report["macro avg"]["recall"],
        "F1_Macro": report["macro avg"]["f1-score"],
    }
    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary_row, f, indent=2)

    print(f"{name} done. Top-1 accuracy: {summary_row['Top1_Accuracy']:.4f}")
    return summary_row

In [ ]:
# =========================================================================
# MAIN
# =========================================================================
def main():
    # Pull class names from the train folder (must match the 10 class folder names exactly)
    class_names = sorted(os.listdir(os.path.join(DATA_DIR, "train")))
    print("Classes found:", class_names)

    all_summaries = [train_custom_cnn(class_names)]
    for name, (base_class, preprocess_fn) in TRANSFER_SPECS.items():
        all_summaries.append(train_transfer_model(name, base_class, preprocess_fn, class_names))

    comparison_df = pd.DataFrame(all_summaries).sort_values("Top1_Accuracy", ascending=False)
    comparison_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison_summary.csv"), index=False)
    print("\nFinal comparison:\n", comparison_df)


if __name__ == "__main__":
    main()

Classes found: ['Basketball', 'BenchPress', 'Biking', 'Fencing', 'GolfSwing', 'HorseRiding', 'Kayaking', 'Skiing', 'SoccerJuggling', 'TennisSwing']

Training: Custom_CNN
Found 6999 images belonging to 10 classes.
Found 1494 images belonging to 10 classes.
Found 1511 images belonging to 10 classes.
Epoch 1/2
219/219 ━━━━━━━━━━━━━━━━━━━━ 3048s 14s/step - accuracy: 0.4083 - loss: 1.6421 - val_accuracy: 0.1794 - val_loss: 3.7099
Epoch 2/2
219/219 ━━━━━━━━━━━━━━━━━━━━ 132s 601ms/step - accuracy: 0.5631 - loss: 1.2104 - val_accuracy: 0.2992 - val_loss: 2.9111


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Custom_CNN done. Top-1 accuracy: 0.2892

Training: VGG16
Found 6999 images belonging to 10 classes.
Found 1494 images belonging to 10 classes.
Found 1511 images belonging to 10 classes.
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/2
219/219 ━━━━━━━━━━━━━━━━━━━━ 199s 829ms/step - accuracy: 0.7047 - loss: 1.1170 - val_accuracy: 0.9203 - val_loss: 0.2476
Epoch 2/2
219/219 ━━━━━━━━━━━━━━━━━━━━ 146s 664ms/step - accuracy: 0.8948 - loss: 0.3151 - val_accuracy: 0.9518 - val_loss: 0.1362
Epoch 1/2
219/219 ━━━━━━━━━━━━━━━━━━━━ 175s 749ms/step - accuracy: 0.9363 - loss: 0.1850 - val_accuracy: 0.9739 - val_loss: 0.0787
Epoch 2/2
219/219 ━━━━━━━━━━━━━━━━━━━━ 161s 733ms/step - accuracy: 0.9646 - loss: 0.1098 - val_accuracy: 0.9853 - val_loss: 0.0462
VGG16 done. Top-1 accuracy: 0.9894

Training: ResNet50
Found 6999 images belonging to 10 classes.
Found 1494 images belonging to 10 classes.
Found 1511 images belonging to 10 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoc

In [ ]:
import os
import re
from collections import defaultdict

DATASET_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset"
SPLITS = ["train", "val", "test"]
CLASSES = ["BenchPress", "Basketball", "Biking", "Fencing", "GolfSwing",
           "HorseRiding", "Kayaking", "Skiing", "SoccerJuggling", "TennisSwing"]

def video_id_from_filename(fname):
    # Filenames were saved as "{original_video_filename}_f{frame_idx}.jpg"
    # Strip the "_f<number>.jpg" suffix to recover the source video's identity.
    return re.sub(r"_f\d+\.jpg$", "", fname)

total_leaked_videos = 0
total_checked_videos = 0

for class_name in CLASSES:
    video_to_splits = defaultdict(set)

    for split in SPLITS:
        class_dir = os.path.join(DATASET_DIR, split, class_name)
        if not os.path.exists(class_dir):
            continue
        for fname in os.listdir(class_dir):
            vid = video_id_from_filename(fname)
            video_to_splits[vid].add(split)

    leaked = {vid: splits for vid, splits in video_to_splits.items() if len(splits) > 1}
    total_checked_videos += len(video_to_splits)
    total_leaked_videos += len(leaked)

    status = "LEAKAGE FOUND" if leaked else "clean"
    print(f"{class_name}: {len(video_to_splits)} source videos, "
          f"{len(leaked)} appear in multiple splits -> {status}")
    if leaked:
        # show up to 2 examples
        for vid, splits in list(leaked.items())[:2]:
            print(f"    e.g. '{vid}' appears in: {splits}")

print(f"\n{'='*60}")
print(f"TOTAL: {total_leaked_videos} / {total_checked_videos} source videos "
      f"have frames split across more than one of train/val/test.")
if total_leaked_videos > 0:
    print("=> Data leakage confirmed. The unrealistically high accuracy is very "
          "likely inflated by this, not a true measure of the model's ability "
          "to generalise to genuinely unseen footage.")
else:
    print("=> No leakage detected by this check. The high accuracy may be genuine "
          "(these classes could just be visually easy to tell apart for ImageNet-"
          "pretrained features) - worth a second look regardless before trusting it fully.")


BenchPress: 160 source videos, 131 appear in multiple splits -> LEAKAGE FOUND
    e.g. 'v_BenchPress_g17_c05' appears in: {'test', 'train', 'val'}
    e.g. 'v_BenchPress_g18_c03' appears in: {'test', 'train', 'val'}
Basketball: 134 source videos, 104 appear in multiple splits -> LEAKAGE FOUND
    e.g. 'v_Basketball_g02_c05' appears in: {'train', 'val'}
    e.g. 'v_Basketball_g05_c01' appears in: {'test', 'train'}
Biking: 134 source videos, 128 appear in multiple splits -> LEAKAGE FOUND
    e.g. 'v_Biking_g15_c05' appears in: {'train', 'val'}
    e.g. 'v_Biking_g06_c04' appears in: {'test', 'train', 'val'}
Fencing: 111 source videos, 101 appear in multiple splits -> LEAKAGE FOUND
    e.g. 'v_Fencing_g03_c05' appears in: {'test', 'train'}
    e.g. 'v_Fencing_g21_c03' appears in: {'train', 'val'}
GolfSwing: 139 source videos, 119 appear in multiple splits -> LEAKAGE FOUND
    e.g. 'v_GolfSwing_g18_c05' appears in: {'test', 'train', 'val'}
    e.g. 'v_GolfSwing_g04_c03' appears in: {'test'

In [ ]:
"""
Fixes the frame-level data leakage found in the original split: groups frames by
their SOURCE VIDEO first, then splits whole videos (never individual frames) into
train/val/test, 70:15:15, stratified by class. This guarantees no video's frames
appear in more than one split.

Run this in Colab (Drive already mounted). Reads from the already-extracted
extracted_frames/ folder in Drive (no need to redo UCF101 download/extraction -
that part was fine; only the splitting step was flawed).
"""

import os
import re
import shutil
import random
from collections import defaultdict

EXTRACTED_FRAMES_DIR = "/content/drive/MyDrive/ucf101_pipeline/extracted_frames"
OUTPUT_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v2"  # new folder - old one left untouched
CLASSES = ["BenchPress", "Basketball", "Biking", "Fencing", "GolfSwing",
           "HorseRiding", "Kayaking", "Skiing", "SoccerJuggling", "TennisSwing"]
SEED = 42
RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}


def video_id_from_filename(fname):
    return re.sub(r"_f\d+\.jpg$", "", fname)


random.seed(SEED)

frame_totals = {"train": 0, "val": 0, "test": 0}
video_totals = {"train": 0, "val": 0, "test": 0}

for class_name in CLASSES:
    class_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)
    if not os.path.exists(class_dir):
        print(f"WARNING: {class_name} not found in extracted_frames - skipping")
        continue

    # Group this class's frames by source video
    video_to_frames = defaultdict(list)
    for fname in os.listdir(class_dir):
        vid = video_id_from_filename(fname)
        video_to_frames[vid].append(fname)

    video_ids = list(video_to_frames.keys())
    random.shuffle(video_ids)  # shuffle at the VIDEO level, not frame level

    n = len(video_ids)
    n_train = int(round(n * RATIOS["train"]))
    n_val = int(round(n * RATIOS["val"]))
    # whatever's left goes to test, avoids rounding losing/gaining a video
    split_videos = {
        "train": video_ids[:n_train],
        "val": video_ids[n_train:n_train + n_val],
        "test": video_ids[n_train + n_val:],
    }

    for split, vids in split_videos.items():
        out_dir = os.path.join(OUTPUT_DIR, split, class_name)
        os.makedirs(out_dir, exist_ok=True)
        n_frames_this_split = 0
        for vid in vids:
            for fname in video_to_frames[vid]:
                shutil.copy(
                    os.path.join(class_dir, fname),
                    os.path.join(out_dir, fname),
                )
                n_frames_this_split += 1
        frame_totals[split] += n_frames_this_split
        video_totals[split] += len(vids)

    print(f"{class_name}: {n} videos -> "
          f"train {len(split_videos['train'])}v, val {len(split_videos['val'])}v, test {len(split_videos['test'])}v")

print(f"\n{'='*60}")
print("FRAME counts per split (should be roughly 70/15/15, may vary a bit "
      "since videos have different frame counts):")
total_frames = sum(frame_totals.values())
for split in ["train", "val", "test"]:
    pct = 100 * frame_totals[split] / total_frames
    print(f"  {split}: {frame_totals[split]} frames ({pct:.1f}%)")

print("\nVIDEO counts per split:")
total_videos = sum(video_totals.values())
for split in ["train", "val", "test"]:
    pct = 100 * video_totals[split] / total_videos
    print(f"  {split}: {video_totals[split]} videos ({pct:.1f}%)")

print(f"\nDone. New leakage-free dataset saved at: {OUTPUT_DIR}")
print("Verify with the leakage-check script pointed at dataset_v2 before training on it.")


BenchPress: 160 videos -> train 112v, val 24v, test 24v
Basketball: 134 videos -> train 94v, val 20v, test 20v
Biking: 134 videos -> train 94v, val 20v, test 20v
Fencing: 111 videos -> train 78v, val 17v, test 16v
GolfSwing: 139 videos -> train 97v, val 21v, test 21v
HorseRiding: 164 videos -> train 115v, val 25v, test 24v
Kayaking: 141 videos -> train 99v, val 21v, test 21v
Skiing: 135 videos -> train 94v, val 20v, test 21v
SoccerJuggling: 147 videos -> train 103v, val 22v, test 22v
TennisSwing: 166 videos -> train 116v, val 25v, test 25v

FRAME counts per split (should be roughly 70/15/15, may vary a bit since videos have different frame counts):
  train: 7066 frames (70.6%)
  val: 1454 frames (14.5%)
  test: 1484 frames (14.8%)

VIDEO counts per split:
  train: 1002 videos (70.0%)
  val: 215 videos (15.0%)
  test: 214 videos (15.0%)

Done. New leakage-free dataset saved at: /content/drive/MyDrive/ucf101_pipeline/dataset_v2
Verify with the leakage-check script pointed at dataset_v2 b

In [ ]:
import os
import re
from collections import defaultdict

DATASET_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v2"   # <-- changed from "dataset"
SPLITS = ["train", "val", "test"]
CLASSES = ["BenchPress", "Basketball", "Biking", "Fencing", "GolfSwing",
           "HorseRiding", "Kayaking", "Skiing", "SoccerJuggling", "TennisSwing"]

def video_id_from_filename(fname):
    return re.sub(r"_f\d+\.jpg$", "", fname)

total_leaked_videos = 0
total_checked_videos = 0

for class_name in CLASSES:
    video_to_splits = defaultdict(set)
    for split in SPLITS:
        class_dir = os.path.join(DATASET_DIR, split, class_name)
        if not os.path.exists(class_dir):
            continue
        for fname in os.listdir(class_dir):
            vid = video_id_from_filename(fname)
            video_to_splits[vid].add(split)

    leaked = {vid: splits for vid, splits in video_to_splits.items() if len(splits) > 1}
    total_checked_videos += len(video_to_splits)
    total_leaked_videos += len(leaked)

    status = "LEAKAGE FOUND" if leaked else "clean"
    print(f"{class_name}: {len(video_to_splits)} source videos, "
          f"{len(leaked)} appear in multiple splits -> {status}")

print(f"\nTOTAL: {total_leaked_videos} / {total_checked_videos} videos leaked across splits.")

BenchPress: 160 source videos, 0 appear in multiple splits -> clean
Basketball: 134 source videos, 0 appear in multiple splits -> clean
Biking: 134 source videos, 0 appear in multiple splits -> clean
Fencing: 111 source videos, 0 appear in multiple splits -> clean
GolfSwing: 139 source videos, 0 appear in multiple splits -> clean
HorseRiding: 164 source videos, 0 appear in multiple splits -> clean
Kayaking: 141 source videos, 0 appear in multiple splits -> clean
Skiing: 135 source videos, 0 appear in multiple splits -> clean
SoccerJuggling: 147 source videos, 0 appear in multiple splits -> clean
TennisSwing: 166 source videos, 0 appear in multiple splits -> clean

TOTAL: 0 / 1431 videos leaked across splits.


In [ ]:

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# =========================================================================
# CONFIG - matches Chapter 3, Sections 3.3 and 3.4
# =========================================================================
DATA_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v2"  # leakage-fixed, video-level split
IMG_SIZE = (224, 224)          # Section 3.3: standard size for VGG16/ResNet50/EfficientNet-B0,
BATCH_SIZE = 32                #   and the custom CNN uses the same resolution for comparability
NUM_CLASSES = 10

# --- QUICK TEST TOGGLE ---
# Set to True first, to confirm the whole pipeline runs end-to-end with tiny
# epoch counts (a few minutes total). Once that works with no errors, set to
# False and re-run for the real Chapter 4 numbers (this will take much longer).
QUICK_TEST = True

if QUICK_TEST:
    CUSTOM_CNN_MAX_EPOCHS = 2
    PHASE1_EPOCHS = 2
    PHASE2_MAX_EPOCHS = 2
else:
    CUSTOM_CNN_MAX_EPOCHS = 50     # Section 3.4.2
    PHASE1_EPOCHS = 20             # Section 3.4.3 - fixed, no early stopping in Phase 1
    PHASE2_MAX_EPOCHS = 30         # ASSUMPTION - Ch3 gives no ceiling, only early stopping

CUSTOM_CNN_LR = 0.001          # Section 3.4.2
CUSTOM_CNN_PATIENCE = 10       # Section 3.4.2 - early stopping on val_loss

PHASE1_LR = 0.001              # ASSUMPTION (see module docstring) - not stated for transfer models
PHASE2_LR = 1e-5               # Section 3.4.3 - explicit
PHASE2_PATIENCE = 10           # ASSUMPTION, mirrors Custom CNN's patience
UNFREEZE_FRACTION = 0.30       # Section 3.4.3 - "top 30% of layers" unfrozen in Phase 2

RESULTS_DIR = "/content/drive/MyDrive/ucf101_pipeline/results_v2"  # NEW folder - keeps old (leaked-data) results separate
SEED = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
tf.random.set_seed(SEED)

In [ ]:
# =========================================================================
# DATA PIPELINE - Section 3.3
# Uses Keras's ImageDataGenerator (named explicitly in Ch3) rather than the
# newer tf.data preprocessing-layers API, to match what Chapter 3 states.
# =========================================================================
# Augmentation parameters exactly as listed in Section 3.3:
#   horizontal flip p=0.5, rotation +/-15deg, width/height shift +/-10%,
#   zoom +/-10%, brightness +/-20%. Applied to the TRAINING set only.
train_datagen_custom = ImageDataGenerator(
    rescale=1.0 / 255,                       # Section 3.3: pixel values normalised to [0,1]
    horizontal_flip=True,                    # p=0.5 is ImageDataGenerator's default flip behaviour
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    brightness_range=(0.8, 1.2),              # +/-20%
)
eval_datagen_custom = ImageDataGenerator(rescale=1.0 / 255)  # val/test: NO augmentation (Section 3.3)


def make_transfer_datagens(preprocess_fn):
    """For the 3 transfer models: same augmentation, but pixel handling goes through
    the model-specific Keras Applications preprocessing function (ImageNet channel-wise
    mean subtraction) instead of a plain /255 rescale - Section 3.3, second paragraph."""
    train_dg = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        horizontal_flip=True,
        rotation_range=15,
        width_shift_range=0.10,
        height_shift_range=0.10,
        zoom_range=0.10,
        brightness_range=(0.8, 1.2),
    )
    eval_dg = ImageDataGenerator(preprocessing_function=preprocess_fn)
    return train_dg, eval_dg


def flow(datagen, split, shuffle):
    return datagen.flow_from_directory(
        os.path.join(DATA_DIR, split),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=shuffle,
        seed=SEED,
    )

In [ ]:
# =========================================================================
# CUSTOM CNN BASELINE - Section 3.4.2, exact architecture
# =========================================================================
def build_custom_cnn():
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, padding="same")(x)   # 3x3 filters
        x = layers.BatchNormalization()(x)                 # BatchNorm after each conv layer
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)                       # 2x2 max-pooling
        return x

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = conv_block(inputs, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256)                                  # filters: 32, 64, 128, 256
    x = layers.GlobalAveragePooling2D()(x)                  # GAP, not Flatten (Section 3.4.2)
    x = layers.Dense(512, activation="relu")(x)             # FC layer, 512 units
    x = layers.Dropout(0.5)(x)                              # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

In [ ]:
# =========================================================================
# TRANSFER LEARNING MODELS - Section 3.4.3, exact head architecture
# =========================================================================
def build_transfer_model(base_class):
    base = base_class(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False  # Phase 1: base fully frozen

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)   # FC layer, 256 units (Section 3.4.3)
    x = layers.Dropout(0.5)(x)                    # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    return model, base


TRANSFER_SPECS = {
    "VGG16": (VGG16, tf.keras.applications.vgg16.preprocess_input),
    "ResNet50": (ResNet50, tf.keras.applications.resnet50.preprocess_input),
    "EfficientNetB0": (EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
}

In [ ]:
# =========================================================================
# TRAIN + EVALUATE: CUSTOM CNN (single phase, Section 3.4.2)
# =========================================================================
# =========================================================================
# RESUME SUPPORT - skip models already fully trained & evaluated in a
# previous session (protects against Colab disconnects mid-run, same idea
# as the resumable dataset-prep script).
# =========================================================================
def already_done(name):
    """If this model's summary.json already exists in RESULTS_DIR, a previous
    session already finished it - return that saved summary instead of
    retraining from scratch. Returns None if not done yet."""
    summary_path = os.path.join(RESULTS_DIR, name, "summary.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            summary = json.load(f)
        print(f"{name}: already completed in a previous session "
              f"(Top-1 accuracy: {summary['Top1_Accuracy']:.4f}) - skipping retraining.")
        return summary
    return None


def train_custom_cnn(class_names):
    name = "Custom_CNN"
    existing = already_done(name)
    if existing is not None:
        return existing

    print(f"\n{'='*70}\nTraining: {name}\n{'='*70}")
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    train_gen = flow(train_datagen_custom, "train", shuffle=True)
    val_gen = flow(eval_datagen_custom, "val", shuffle=True)
    test_gen = flow(eval_datagen_custom, "test", shuffle=False)

    model = build_custom_cnn()
    model.compile(
        optimizer=optimizers.Adam(learning_rate=CUSTOM_CNN_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    ckpt_path = os.path.join(out_dir, "best_model.keras")
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=CUSTOM_CNN_PATIENCE, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase1.csv")),
    ]
    model.fit(train_gen, validation_data=val_gen, epochs=CUSTOM_CNN_MAX_EPOCHS, callbacks=callbacks)

    model = tf.keras.models.load_model(ckpt_path)
    return evaluate_and_save(model, test_gen, class_names, out_dir, name)

In [ ]:
# =========================================================================
# TRAIN + EVALUATE: ONE TRANSFER MODEL (two phases, Section 3.4.3)
# =========================================================================
def train_transfer_model(name, base_class, preprocess_fn, class_names):
    existing = already_done(name)
    if existing is not None:
        return existing

    print(f"\n{'='*70}\nTraining: {name}\n{'='*70}")
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    train_dg, eval_dg = make_transfer_datagens(preprocess_fn)
    train_gen = flow(train_dg, "train", shuffle=True)
    val_gen = flow(eval_dg, "val", shuffle=True)
    test_gen = flow(eval_dg, "test", shuffle=False)

    model, base = build_transfer_model(base_class)
    ckpt_path = os.path.join(out_dir, "best_model.keras")

    # ---------------- PHASE 1: frozen base, head only, 20 fixed epochs, NO early stopping ----------------
    model.compile(optimizer=optimizers.Adam(learning_rate=PHASE1_LR),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        train_gen, validation_data=val_gen, epochs=PHASE1_EPOCHS,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
            tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase1.csv")),
        ],
    )

    # ---------------- PHASE 2: unfreeze top 30% of base, LR=1e-5, early stop on val_loss ----------------
    base.trainable = True
    freeze_until = int(len(base.layers) * (1 - UNFREEZE_FRACTION))
    for layer in base.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(learning_rate=PHASE2_LR),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        train_gen, validation_data=val_gen, epochs=PHASE2_MAX_EPOCHS,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PHASE2_PATIENCE, restore_best_weights=True),
            tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase2.csv")),
        ],
    )

    model = tf.keras.models.load_model(ckpt_path)
    return evaluate_and_save(model, test_gen, class_names, out_dir, name)

In [ ]:
# =========================================================================
# SHARED EVALUATION - Section 3.5 (accuracy, macro P/R/F1, confusion matrix)
# =========================================================================
def evaluate_and_save(model, test_gen, class_names, out_dir, name):
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes  # ground-truth labels in the (unshuffled) test generator order

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(os.path.join(out_dir, "classification_report.csv"))

    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(os.path.join(out_dir, "confusion_matrix.csv"))

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.close()

    summary_row = {
        "Model": name,
        "Top1_Accuracy": report["accuracy"],
        "Precision_Macro": report["macro avg"]["precision"],
        "Recall_Macro": report["macro avg"]["recall"],
        "F1_Macro": report["macro avg"]["f1-score"],
    }
    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary_row, f, indent=2)

    print(f"{name} done. Top-1 accuracy: {summary_row['Top1_Accuracy']:.4f}")
    return summary_row

In [ ]:
# =========================================================================
# MAIN
# =========================================================================
def main():
    # Pull class names from the train folder (must match the 10 class folder names exactly)
    class_names = sorted(os.listdir(os.path.join(DATA_DIR, "train")))
    print("Classes found:", class_names)

    all_summaries = [train_custom_cnn(class_names)]
    for name, (base_class, preprocess_fn) in TRANSFER_SPECS.items():
        all_summaries.append(train_transfer_model(name, base_class, preprocess_fn, class_names))

    comparison_df = pd.DataFrame(all_summaries).sort_values("Top1_Accuracy", ascending=False)
    comparison_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison_summary.csv"), index=False)
    print("\nFinal comparison:\n", comparison_df)


if __name__ == "__main__":
    main()

Classes found: ['Basketball', 'BenchPress', 'Biking', 'Fencing', 'GolfSwing', 'HorseRiding', 'Kayaking', 'Skiing', 'SoccerJuggling', 'TennisSwing']

Training: Custom_CNN
Found 7066 images belonging to 10 classes.
Found 1454 images belonging to 10 classes.
Found 1484 images belonging to 10 classes.
Epoch 1/2
221/221 ━━━━━━━━━━━━━━━━━━━━ 156s 662ms/step - accuracy: 0.4192 - loss: 1.6247 - val_accuracy: 0.1382 - val_loss: 3.9593
Epoch 2/2
221/221 ━━━━━━━━━━━━━━━━━━━━ 137s 621ms/step - accuracy: 0.5759 - loss: 1.1752 - val_accuracy: 0.3838 - val_loss: 1.9613
Custom_CNN done. Top-1 accuracy: 0.3679

Training: VGG16
Found 7066 images belonging to 10 classes.
Found 1454 images belonging to 10 classes.
Found 1484 images belonging to 10 classes.
Epoch 1/2
221/221 ━━━━━━━━━━━━━━━━━━━━ 176s 782ms/step - accuracy: 0.7274 - loss: 0.9546 - val_accuracy: 0.9023 - val_loss: 0.2976
Epoch 2/2
221/221 ━━━━━━━━━━━━━━━━━━━━ 154s 696ms/step - accuracy: 0.8994 - loss: 0.2991 - val_accuracy: 0.9367 - val_loss

In [ ]:
import os
import re
from collections import defaultdict

DATASET_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v2"
SPLITS = ["train", "val", "test"]
CLASSES = ["BenchPress", "Basketball", "Biking", "Fencing", "GolfSwing",
           "HorseRiding", "Kayaking", "Skiing", "SoccerJuggling", "TennisSwing"]

# UCF101 filenames look like: v_BenchPress_g17_c05_f3.jpg
# "g17" = group number (same actor/background/session) - shared across c01-c07 clips
GROUP_PATTERN = re.compile(r"^(v_\w+?_g\d+)_c\d+_f\d+\.jpg$")

def group_id_from_filename(fname):
    m = GROUP_PATTERN.match(fname)
    return m.group(1) if m else None

total_leaked_groups = 0
total_checked_groups = 0
unparsed = 0

for class_name in CLASSES:
    group_to_splits = defaultdict(set)

    for split in SPLITS:
        class_dir = os.path.join(DATASET_DIR, split, class_name)
        if not os.path.exists(class_dir):
            continue
        for fname in os.listdir(class_dir):
            gid = group_id_from_filename(fname)
            if gid is None:
                globals()["unparsed"] = unparsed + 1
                continue
            group_to_splits[gid].add(split)

    leaked = {gid: splits for gid, splits in group_to_splits.items() if len(splits) > 1}
    total_checked_groups += len(group_to_splits)
    total_leaked_groups += len(leaked)

    status = "GROUP LEAKAGE FOUND" if leaked else "clean"
    print(f"{class_name}: {len(group_to_splits)} groups, "
          f"{len(leaked)} appear in multiple splits -> {status}")
    if leaked:
        for gid, splits in list(leaked.items())[:2]:
            print(f"    e.g. '{gid}' appears in: {splits}")

print(f"\n{'='*60}")
print(f"TOTAL: {total_leaked_groups} / {total_checked_groups} groups leaked across splits.")
if unparsed:
    print(f"NOTE: {unparsed} filenames didn't match the expected UCF101 naming pattern "
          f"and were skipped - check these manually if this number is large.")
if total_leaked_groups > 0:
    print("=> Group-level leakage present. Same actor/background appears in more than "
          "one split, which can still inflate accuracy somewhat (milder than the "
          "frame-level issue, but worth fixing for a rigorous split).")
else:
    print("=> Clean at the group level too. This is now a properly rigorous split.")

BenchPress: 25 groups, 24 appear in multiple splits -> GROUP LEAKAGE FOUND
    e.g. 'v_BenchPress_g17' appears in: {'train', 'val'}
    e.g. 'v_BenchPress_g22' appears in: {'train', 'val'}
Basketball: 25 groups, 23 appear in multiple splits -> GROUP LEAKAGE FOUND
    e.g. 'v_Basketball_g17' appears in: {'test', 'train'}
    e.g. 'v_Basketball_g13' appears in: {'train', 'val'}
Biking: 25 groups, 21 appear in multiple splits -> GROUP LEAKAGE FOUND
    e.g. 'v_Biking_g11' appears in: {'test', 'train', 'val'}
    e.g. 'v_Biking_g21' appears in: {'test', 'train', 'val'}
Fencing: 25 groups, 19 appear in multiple splits -> GROUP LEAKAGE FOUND
    e.g. 'v_Fencing_g01' appears in: {'train', 'val'}
    e.g. 'v_Fencing_g05' appears in: {'train', 'val'}
GolfSwing: 25 groups, 22 appear in multiple splits -> GROUP LEAKAGE FOUND
    e.g. 'v_GolfSwing_g17' appears in: {'test', 'train', 'val'}
    e.g. 'v_GolfSwing_g05' appears in: {'test', 'train', 'val'}
HorseRiding: 25 groups, 22 appear in multiple 

In [ ]:
"""
Final, correct split: groups frames by UCF101's own GROUP number (v_ClassName_gNN),
which represents a shared actor/background/filming session. All clips (c01-c07)
within a group are kept together in the SAME split - this is the leakage-prevention
mechanism UCF101 itself was designed around, so this is the last level that needs
fixing (no finer-grained leakage exists below "group").

Run in Colab (Drive already mounted). Reads from extracted_frames/ (unchanged,
still the original 1fps-sampled images - only the splitting logic changes again).
"""

import os
import re
import shutil
import random
from collections import defaultdict

EXTRACTED_FRAMES_DIR = "/content/drive/MyDrive/ucf101_pipeline/extracted_frames"
OUTPUT_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v3"  # new folder, v2 left untouched
CLASSES = ["BenchPress", "Basketball", "Biking", "Fencing", "GolfSwing",
           "HorseRiding", "Kayaking", "Skiing", "SoccerJuggling", "TennisSwing"]
SEED = 42
RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}

GROUP_PATTERN = re.compile(r"^(v_\w+?_g\d+)_c\d+_f\d+\.jpg$")

def group_id_from_filename(fname):
    m = GROUP_PATTERN.match(fname)
    return m.group(1) if m else None


random.seed(SEED)

frame_totals = {"train": 0, "val": 0, "test": 0}
group_totals = {"train": 0, "val": 0, "test": 0}

for class_name in CLASSES:
    class_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)
    if not os.path.exists(class_dir):
        print(f"WARNING: {class_name} not found in extracted_frames - skipping")
        continue

    # Group this class's frames by UCF101 group id (not individual video this time)
    group_to_frames = defaultdict(list)
    unparsed = 0
    for fname in os.listdir(class_dir):
        gid = group_id_from_filename(fname)
        if gid is None:
            unparsed += 1
            continue
        group_to_frames[gid].append(fname)
    if unparsed:
        print(f"  ({class_name}: {unparsed} filenames didn't match the expected pattern, skipped)")

    group_ids = list(group_to_frames.keys())
    random.shuffle(group_ids)  # shuffle at the GROUP level

    n = len(group_ids)
    n_train = int(round(n * RATIOS["train"]))
    n_val = int(round(n * RATIOS["val"]))
    split_groups = {
        "train": group_ids[:n_train],
        "val": group_ids[n_train:n_train + n_val],
        "test": group_ids[n_train + n_val:],
    }

    for split, gids in split_groups.items():
        out_dir = os.path.join(OUTPUT_DIR, split, class_name)
        os.makedirs(out_dir, exist_ok=True)
        n_frames_this_split = 0
        for gid in gids:
            for fname in group_to_frames[gid]:
                shutil.copy(
                    os.path.join(class_dir, fname),
                    os.path.join(out_dir, fname),
                )
                n_frames_this_split += 1
        frame_totals[split] += n_frames_this_split
        group_totals[split] += len(gids)

    print(f"{class_name}: {n} groups -> "
          f"train {len(split_groups['train'])}g, val {len(split_groups['val'])}g, test {len(split_groups['test'])}g")

print(f"\n{'='*60}")
print("FRAME counts per split:")
total_frames = sum(frame_totals.values())
for split in ["train", "val", "test"]:
    pct = 100 * frame_totals[split] / total_frames
    print(f"  {split}: {frame_totals[split]} frames ({pct:.1f}%)")

print("\nGROUP counts per split:")
total_groups = sum(group_totals.values())
for split in ["train", "val", "test"]:
    pct = 100 * group_totals[split] / total_groups
    print(f"  {split}: {group_totals[split]} groups ({pct:.1f}%)")

print(f"\nDone. Group-level leakage-free dataset saved at: {OUTPUT_DIR}")
print("Verify with the group-leakage-check script (pointed at dataset_v3) before training on it.")

BenchPress: 25 groups -> train 18g, val 4g, test 3g
Basketball: 25 groups -> train 18g, val 4g, test 3g
Biking: 25 groups -> train 18g, val 4g, test 3g
Fencing: 25 groups -> train 18g, val 4g, test 3g
GolfSwing: 25 groups -> train 18g, val 4g, test 3g
HorseRiding: 25 groups -> train 18g, val 4g, test 3g
Kayaking: 25 groups -> train 18g, val 4g, test 3g
Skiing: 25 groups -> train 18g, val 4g, test 3g
SoccerJuggling: 25 groups -> train 18g, val 4g, test 3g
TennisSwing: 25 groups -> train 18g, val 4g, test 3g

FRAME counts per split:
  train: 7202 frames (72.0%)
  val: 1607 frames (16.1%)
  test: 1195 frames (11.9%)

GROUP counts per split:
  train: 180 groups (72.0%)
  val: 40 groups (16.0%)
  test: 30 groups (12.0%)

Done. Group-level leakage-free dataset saved at: /content/drive/MyDrive/ucf101_pipeline/dataset_v3
Verify with the group-leakage-check script (pointed at dataset_v3) before training on it.


In [ ]:
import os
import re
from collections import defaultdict

DATASET_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v3"
SPLITS = ["train", "val", "test"]
CLASSES = ["BenchPress", "Basketball", "Biking", "Fencing", "GolfSwing",
           "HorseRiding", "Kayaking", "Skiing", "SoccerJuggling", "TennisSwing"]

# UCF101 filenames look like: v_BenchPress_g17_c05_f3.jpg
# "g17" = group number (same actor/background/session) - shared across c01-c07 clips
GROUP_PATTERN = re.compile(r"^(v_\w+?_g\d+)_c\d+_f\d+\.jpg$")

def group_id_from_filename(fname):
    m = GROUP_PATTERN.match(fname)
    return m.group(1) if m else None

total_leaked_groups = 0
total_checked_groups = 0
unparsed = 0

for class_name in CLASSES:
    group_to_splits = defaultdict(set)

    for split in SPLITS:
        class_dir = os.path.join(DATASET_DIR, split, class_name)
        if not os.path.exists(class_dir):
            continue
        for fname in os.listdir(class_dir):
            gid = group_id_from_filename(fname)
            if gid is None:
                unparsed += 1
                continue
            group_to_splits[gid].add(split)

    leaked = {gid: splits for gid, splits in group_to_splits.items() if len(splits) > 1}
    total_checked_groups += len(group_to_splits)
    total_leaked_groups += len(leaked)

    status = "GROUP LEAKAGE FOUND" if leaked else "clean"
    print(f"{class_name}: {len(group_to_splits)} groups, "
          f"{len(leaked)} appear in multiple splits -> {status}")
    if leaked:
        for gid, splits in list(leaked.items())[:2]:
            print(f"    e.g. '{gid}' appears in: {splits}")

print(f"\n{'='*60}")
print(f"TOTAL: {total_leaked_groups} / {total_checked_groups} groups leaked across splits.")
if unparsed:
    print(f"NOTE: {unparsed} filenames didn't match the expected UCF101 naming pattern "
          f"and were skipped - check these manually if this number is large.")
if total_leaked_groups > 0:
    print("=> Group-level leakage still present.")
else:
    print("=> Clean at the group level. This split is now rigorous.")


BenchPress: 25 groups, 0 appear in multiple splits -> clean
Basketball: 25 groups, 0 appear in multiple splits -> clean
Biking: 25 groups, 0 appear in multiple splits -> clean
Fencing: 25 groups, 0 appear in multiple splits -> clean
GolfSwing: 25 groups, 0 appear in multiple splits -> clean
HorseRiding: 25 groups, 0 appear in multiple splits -> clean
Kayaking: 25 groups, 0 appear in multiple splits -> clean
Skiing: 25 groups, 0 appear in multiple splits -> clean
SoccerJuggling: 25 groups, 0 appear in multiple splits -> clean
TennisSwing: 25 groups, 0 appear in multiple splits -> clean

TOTAL: 0 / 250 groups leaked across splits.
=> Clean at the group level. This split is now rigorous.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# =========================================================================
# CONFIG - matches Chapter 3, Sections 3.3 and 3.4
# =========================================================================
DATA_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v3"  # group-level leakage-fixed (final)
IMG_SIZE = (224, 224)          # Section 3.3: standard size for VGG16/ResNet50/EfficientNet-B0,
BATCH_SIZE = 32                #   and the custom CNN uses the same resolution for comparability
NUM_CLASSES = 10

# --- QUICK TEST TOGGLE ---
# Set to True first, to confirm the whole pipeline runs end-to-end with tiny
# epoch counts (a few minutes total). Once that works with no errors, set to
# False and re-run for the real Chapter 4 numbers (this will take much longer).
QUICK_TEST = True

if QUICK_TEST:
    CUSTOM_CNN_MAX_EPOCHS = 2
    PHASE1_EPOCHS = 2
    PHASE2_MAX_EPOCHS = 2
else:
    CUSTOM_CNN_MAX_EPOCHS = 50     # Section 3.4.2
    PHASE1_EPOCHS = 20             # Section 3.4.3 - fixed, no early stopping in Phase 1
    PHASE2_MAX_EPOCHS = 30         # ASSUMPTION - Ch3 gives no ceiling, only early stopping

CUSTOM_CNN_LR = 0.001          # Section 3.4.2
CUSTOM_CNN_PATIENCE = 10       # Section 3.4.2 - early stopping on val_loss

PHASE1_LR = 0.001              # ASSUMPTION (see module docstring) - not stated for transfer models
PHASE2_LR = 1e-5               # Section 3.4.3 - explicit
PHASE2_PATIENCE = 10           # ASSUMPTION, mirrors Custom CNN's patience
UNFREEZE_FRACTION = 0.30       # Section 3.4.3 - "top 30% of layers" unfrozen in Phase 2

RESULTS_DIR = "/content/drive/MyDrive/ucf101_pipeline/results_v3"  # NEW folder - keeps results_v2 (video-level, still had group leakage) separate
SEED = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
tf.random.set_seed(SEED)

In [ ]:
# =========================================================================
# DATA PIPELINE - Section 3.3
# Uses Keras's ImageDataGenerator (named explicitly in Ch3) rather than the
# newer tf.data preprocessing-layers API, to match what Chapter 3 states.
# =========================================================================
# Augmentation parameters exactly as listed in Section 3.3:
#   horizontal flip p=0.5, rotation +/-15deg, width/height shift +/-10%,
#   zoom +/-10%, brightness +/-20%. Applied to the TRAINING set only.
train_datagen_custom = ImageDataGenerator(
    rescale=1.0 / 255,                       # Section 3.3: pixel values normalised to [0,1]
    horizontal_flip=True,                    # p=0.5 is ImageDataGenerator's default flip behaviour
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    brightness_range=(0.8, 1.2),              # +/-20%
)
eval_datagen_custom = ImageDataGenerator(rescale=1.0 / 255)  # val/test: NO augmentation (Section 3.3)


def make_transfer_datagens(preprocess_fn):
    """For the 3 transfer models: same augmentation, but pixel handling goes through
    the model-specific Keras Applications preprocessing function (ImageNet channel-wise
    mean subtraction) instead of a plain /255 rescale - Section 3.3, second paragraph."""
    train_dg = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        horizontal_flip=True,
        rotation_range=15,
        width_shift_range=0.10,
        height_shift_range=0.10,
        zoom_range=0.10,
        brightness_range=(0.8, 1.2),
    )
    eval_dg = ImageDataGenerator(preprocessing_function=preprocess_fn)
    return train_dg, eval_dg


def flow(datagen, split, shuffle):
    return datagen.flow_from_directory(
        os.path.join(DATA_DIR, split),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=shuffle,
        seed=SEED,
    )

In [ ]:
# =========================================================================
# CUSTOM CNN BASELINE - Section 3.4.2, exact architecture
# =========================================================================
def build_custom_cnn():
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, padding="same")(x)   # 3x3 filters
        x = layers.BatchNormalization()(x)                 # BatchNorm after each conv layer
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)                       # 2x2 max-pooling
        return x

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = conv_block(inputs, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256)                                  # filters: 32, 64, 128, 256
    x = layers.GlobalAveragePooling2D()(x)                  # GAP, not Flatten (Section 3.4.2)
    x = layers.Dense(512, activation="relu")(x)             # FC layer, 512 units
    x = layers.Dropout(0.5)(x)                              # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

In [ ]:
# =========================================================================
# TRANSFER LEARNING MODELS - Section 3.4.3, exact head architecture
# =========================================================================
def build_transfer_model(base_class):
    base = base_class(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False  # Phase 1: base fully frozen

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)   # FC layer, 256 units (Section 3.4.3)
    x = layers.Dropout(0.5)(x)                    # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    return model, base


TRANSFER_SPECS = {
    "VGG16": (VGG16, tf.keras.applications.vgg16.preprocess_input),
    "ResNet50": (ResNet50, tf.keras.applications.resnet50.preprocess_input),
    "EfficientNetB0": (EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
}

In [ ]:
# =========================================================================
# TRAIN + EVALUATE: CUSTOM CNN (single phase, Section 3.4.2)
# =========================================================================
# =========================================================================
# RESUME SUPPORT - skip models already fully trained & evaluated in a
# previous session (protects against Colab disconnects mid-run, same idea
# as the resumable dataset-prep script).
# =========================================================================
def already_done(name):
    """If this model's summary.json already exists in RESULTS_DIR, a previous
    session already finished it - return that saved summary instead of
    retraining from scratch. Returns None if not done yet."""
    summary_path = os.path.join(RESULTS_DIR, name, "summary.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            summary = json.load(f)
        print(f"{name}: already completed in a previous session "
              f"(Top-1 accuracy: {summary['Top1_Accuracy']:.4f}) - skipping retraining.")
        return summary
    return None


def train_custom_cnn(class_names):
    name = "Custom_CNN"
    existing = already_done(name)
    if existing is not None:
        return existing

    print(f"\n{'='*70}\nTraining: {name}\n{'='*70}")
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    train_gen = flow(train_datagen_custom, "train", shuffle=True)
    val_gen = flow(eval_datagen_custom, "val", shuffle=True)
    test_gen = flow(eval_datagen_custom, "test", shuffle=False)

    model = build_custom_cnn()
    model.compile(
        optimizer=optimizers.Adam(learning_rate=CUSTOM_CNN_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    ckpt_path = os.path.join(out_dir, "best_model.keras")
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=CUSTOM_CNN_PATIENCE, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase1.csv")),
    ]
    model.fit(train_gen, validation_data=val_gen, epochs=CUSTOM_CNN_MAX_EPOCHS, callbacks=callbacks)

    model = tf.keras.models.load_model(ckpt_path)
    return evaluate_and_save(model, test_gen, class_names, out_dir, name)

In [ ]:
# =========================================================================
# SHARED EVALUATION - Section 3.5 (accuracy, macro P/R/F1, confusion matrix)
# =========================================================================
def evaluate_and_save(model, test_gen, class_names, out_dir, name):
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes  # ground-truth labels in the (unshuffled) test generator order

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(os.path.join(out_dir, "classification_report.csv"))

    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(os.path.join(out_dir, "confusion_matrix.csv"))

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.close()

    summary_row = {
        "Model": name,
        "Top1_Accuracy": report["accuracy"],
        "Precision_Macro": report["macro avg"]["precision"],
        "Recall_Macro": report["macro avg"]["recall"],
        "F1_Macro": report["macro avg"]["f1-score"],
    }
    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary_row, f, indent=2)

    print(f"{name} done. Top-1 accuracy: {summary_row['Top1_Accuracy']:.4f}")
    return summary_row

In [ ]:
# =========================================================================
# MAIN
# =========================================================================
def main():
    # Pull class names from the train folder (must match the 10 class folder names exactly)
    class_names = sorted(os.listdir(os.path.join(DATA_DIR, "train")))
    print("Classes found:", class_names)

    all_summaries = [train_custom_cnn(class_names)]
    for name, (base_class, preprocess_fn) in TRANSFER_SPECS.items():
        all_summaries.append(train_transfer_model(name, base_class, preprocess_fn, class_names))

    comparison_df = pd.DataFrame(all_summaries).sort_values("Top1_Accuracy", ascending=False)
    comparison_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison_summary.csv"), index=False)
    print("\nFinal comparison:\n", comparison_df)


if __name__ == "__main__":
    main()

Classes found: ['Basketball', 'BenchPress', 'Biking', 'Fencing', 'GolfSwing', 'HorseRiding', 'Kayaking', 'Skiing', 'SoccerJuggling', 'TennisSwing']

Training: Custom_CNN
Found 7202 images belonging to 10 classes.
Found 1607 images belonging to 10 classes.
Found 1195 images belonging to 10 classes.
Epoch 1/2
226/226 ━━━━━━━━━━━━━━━━━━━━ 168s 695ms/step - accuracy: 0.4274 - loss: 1.5984 - val_accuracy: 0.2035 - val_loss: 3.8204
Epoch 2/2
226/226 ━━━━━━━━━━━━━━━━━━━━ 148s 656ms/step - accuracy: 0.6077 - loss: 1.1169 - val_accuracy: 0.3180 - val_loss: 2.8063


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Custom_CNN done. Top-1 accuracy: 0.2209

Training: VGG16
Found 7202 images belonging to 10 classes.
Found 1607 images belonging to 10 classes.
Found 1195 images belonging to 10 classes.
Epoch 1/2
226/226 ━━━━━━━━━━━━━━━━━━━━ 173s 750ms/step - accuracy: 0.7430 - loss: 0.9171 - val_accuracy: 0.8656 - val_loss: 0.3778
Epoch 2/2
226/226 ━━━━━━━━━━━━━━━━━━━━ 164s 724ms/step - accuracy: 0.9084 - loss: 0.2688 - val_accuracy: 0.8519 - val_loss: 0.5086
Epoch 1/2
226/226 ━━━━━━━━━━━━━━━━━━━━ 181s 770ms/step - accuracy: 0.9542 - loss: 0.1380 - val_accuracy: 0.8737 - val_loss: 0.4571
Epoch 2/2
226/226 ━━━━━━━━━━━━━━━━━━━━ 169s 745ms/step - accuracy: 0.9753 - loss: 0.0776 - val_accuracy: 0.8718 - val_loss: 0.6081
VGG16 done. Top-1 accuracy: 0.8611

Training: ResNet50
Found 7202 images belonging to 10 classes.
Found 1607 images belonging to 10 classes.
Found 1195 images belonging to 10 classes.
Epoch 1/2
226/226 ━━━━━━━━━━━━━━━━━━━━ 173s 710ms/step - accuracy: 0.8321 - loss: 0.5057 - val_accuracy: 0

In [ ]:
"""
Two-phase fine-tuning pipeline for game action recognition (10-class UCF101 subset).
Rebuilt to match Chapter 3 (Methodology) EXACTLY - see inline comments citing the
relevant Chapter 3 section for every design choice.

Folder structure expected (already split 70:15:15, stratified by class):
dataset/
    train/<class_name>/*.jpg
    val/<class_name>/*.jpg
    test/<class_name>/*.jpg

Everything needed for Chapter 4 gets written to results/<model_name>/:
    - training_log_phase1.csv, training_log_phase2.csv   (per-epoch acc/loss -> training curves, Section 3.5)
    - classification_report.csv                          (accuracy, macro precision/recall/F1, Section 3.5)
    - confusion_matrix.csv + confusion_matrix.png         (10x10, Section 3.5)
    - best_model.keras
And a combined results/model_comparison_summary.csv across all 4 models.

ASSUMPTIONS MADE where Chapter 3 does not give an exact number (flagged so you can
adjust or add the missing number into Chapter 3 for consistency):
    - Phase 1 learning rate for the 3 transfer-learning models: not stated in 3.4.3
      (only the Phase 2 rate of 1e-5 is given). Set to 0.001, matching the Custom
      CNN's stated rate and standard practice for frozen-base training.
    - Phase 2 max epoch cap for transfer-learning models: 3.4.3 says early stopping
      on val_loss governs Phase 2 but gives no epoch ceiling. Set to 30 (generous
      ceiling; early stopping will normally end it sooner).
    - Early stopping patience in Phase 2 for transfer models: not stated. Set to 10,
      matching the Custom CNN's stated patience (Section 3.4.2).
"""
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns




In [ ]:
# =========================================================================
# CONFIG - matches Chapter 3, Sections 3.3 and 3.4
# =========================================================================
DATA_DIR = "/content/drive/MyDrive/ucf101_pipeline/dataset_v3"  # group-level leakage-fixed (final)
IMG_SIZE = (224, 224)          # Section 3.3: standard size for VGG16/ResNet50/EfficientNet-B0,
BATCH_SIZE = 32                #   and the custom CNN uses the same resolution for comparability
NUM_CLASSES = 10

# --- QUICK TEST TOGGLE ---
# Set to True first, to confirm the whole pipeline runs end-to-end with tiny
# epoch counts (a few minutes total). Once that works with no errors, set to
# False and re-run for the real Chapter 4 numbers (this will take much longer).
QUICK_TEST = False

if QUICK_TEST:
    CUSTOM_CNN_MAX_EPOCHS = 2
    PHASE1_EPOCHS = 2
    PHASE2_MAX_EPOCHS = 2
else:
    CUSTOM_CNN_MAX_EPOCHS = 50     # Section 3.4.2
    PHASE1_EPOCHS = 20             # Section 3.4.3 - fixed, no early stopping in Phase 1
    PHASE2_MAX_EPOCHS = 30         # ASSUMPTION - Ch3 gives no ceiling, only early stopping

CUSTOM_CNN_LR = 0.001          # Section 3.4.2
CUSTOM_CNN_PATIENCE = 10       # Section 3.4.2 - early stopping on val_loss

PHASE1_LR = 0.001              # ASSUMPTION (see module docstring) - not stated for transfer models
PHASE2_LR = 1e-5               # Section 3.4.3 - explicit
PHASE2_PATIENCE = 10           # ASSUMPTION, mirrors Custom CNN's patience
UNFREEZE_FRACTION = 0.30       # Section 3.4.3 - "top 30% of layers" unfrozen in Phase 2

RESULTS_DIR = "/content/drive/MyDrive/ucf101_pipeline/results_v3_full"  # NEW folder - the quick test's 2-epoch results in results_v3 must NOT be reused here
SEED = 42

os.makedirs(RESULTS_DIR, exist_ok=True)
tf.random.set_seed(SEED)

In [ ]:
# =========================================================================
# DATA PIPELINE - Section 3.3
# Uses Keras's ImageDataGenerator (named explicitly in Ch3) rather than the
# newer tf.data preprocessing-layers API, to match what Chapter 3 states.
# =========================================================================
# Augmentation parameters exactly as listed in Section 3.3:
#   horizontal flip p=0.5, rotation +/-15deg, width/height shift +/-10%,
#   zoom +/-10%, brightness +/-20%. Applied to the TRAINING set only.
train_datagen_custom = ImageDataGenerator(
    rescale=1.0 / 255,                       # Section 3.3: pixel values normalised to [0,1]
    horizontal_flip=True,                    # p=0.5 is ImageDataGenerator's default flip behaviour
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    brightness_range=(0.8, 1.2),              # +/-20%
)
eval_datagen_custom = ImageDataGenerator(rescale=1.0 / 255)  # val/test: NO augmentation (Section 3.3)


def make_transfer_datagens(preprocess_fn):
    """For the 3 transfer models: same augmentation, but pixel handling goes through
    the model-specific Keras Applications preprocessing function (ImageNet channel-wise
    mean subtraction) instead of a plain /255 rescale - Section 3.3, second paragraph."""
    train_dg = ImageDataGenerator(
        preprocessing_function=preprocess_fn,
        horizontal_flip=True,
        rotation_range=15,
        width_shift_range=0.10,
        height_shift_range=0.10,
        zoom_range=0.10,
        brightness_range=(0.8, 1.2),
    )
    eval_dg = ImageDataGenerator(preprocessing_function=preprocess_fn)
    return train_dg, eval_dg


def flow(datagen, split, shuffle):
    return datagen.flow_from_directory(
        os.path.join(DATA_DIR, split),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=shuffle,
        seed=SEED,
    )


In [ ]:
# =========================================================================
# CUSTOM CNN BASELINE - Section 3.4.2, exact architecture
# =========================================================================
def build_custom_cnn():
    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, padding="same")(x)   # 3x3 filters
        x = layers.BatchNormalization()(x)                 # BatchNorm after each conv layer
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D(2)(x)                       # 2x2 max-pooling
        return x

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = conv_block(inputs, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)
    x = conv_block(x, 256)                                  # filters: 32, 64, 128, 256
    x = layers.GlobalAveragePooling2D()(x)                  # GAP, not Flatten (Section 3.4.2)
    x = layers.Dense(512, activation="relu")(x)             # FC layer, 512 units
    x = layers.Dropout(0.5)(x)                              # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

In [ ]:
# =========================================================================
# TRANSFER LEARNING MODELS - Section 3.4.3, exact head architecture
# =========================================================================
def build_transfer_model(base_class):
    base = base_class(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False  # Phase 1: base fully frozen

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)   # FC layer, 256 units (Section 3.4.3)
    x = layers.Dropout(0.5)(x)                    # dropout 0.5
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs)
    return model, base


TRANSFER_SPECS = {
    "VGG16": (VGG16, tf.keras.applications.vgg16.preprocess_input),
    "ResNet50": (ResNet50, tf.keras.applications.resnet50.preprocess_input),
    "EfficientNetB0": (EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
}

In [ ]:
# =========================================================================
# TRAIN + EVALUATE: CUSTOM CNN (single phase, Section 3.4.2)
# =========================================================================
# =========================================================================
# RESUME SUPPORT - skip models already fully trained & evaluated in a
# previous session (protects against Colab disconnects mid-run, same idea
# as the resumable dataset-prep script).
# =========================================================================
def already_done(name):
    """If this model's summary.json already exists in RESULTS_DIR, a previous
    session already finished it - return that saved summary instead of
    retraining from scratch. Returns None if not done yet."""
    summary_path = os.path.join(RESULTS_DIR, name, "summary.json")
    if os.path.exists(summary_path):
        with open(summary_path) as f:
            summary = json.load(f)
        print(f"{name}: already completed in a previous session "
              f"(Top-1 accuracy: {summary['Top1_Accuracy']:.4f}) - skipping retraining.")
        return summary
    return None


def train_custom_cnn(class_names):
    name = "Custom_CNN"
    existing = already_done(name)
    if existing is not None:
        return existing

    print(f"\n{'='*70}\nTraining: {name}\n{'='*70}")
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    train_gen = flow(train_datagen_custom, "train", shuffle=True)
    val_gen = flow(eval_datagen_custom, "val", shuffle=True)
    test_gen = flow(eval_datagen_custom, "test", shuffle=False)

    model = build_custom_cnn()
    model.compile(
        optimizer=optimizers.Adam(learning_rate=CUSTOM_CNN_LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    ckpt_path = os.path.join(out_dir, "best_model.keras")
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=CUSTOM_CNN_PATIENCE, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase1.csv")),
    ]
    model.fit(train_gen, validation_data=val_gen, epochs=CUSTOM_CNN_MAX_EPOCHS, callbacks=callbacks)

    model = tf.keras.models.load_model(ckpt_path)
    return evaluate_and_save(model, test_gen, class_names, out_dir, name)


In [ ]:
# =========================================================================
# TRAIN + EVALUATE: ONE TRANSFER MODEL (two phases, Section 3.4.3)
# =========================================================================
def train_transfer_model(name, base_class, preprocess_fn, class_names):
    existing = already_done(name)
    if existing is not None:
        return existing

    print(f"\n{'='*70}\nTraining: {name}\n{'='*70}")
    out_dir = os.path.join(RESULTS_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    train_dg, eval_dg = make_transfer_datagens(preprocess_fn)
    train_gen = flow(train_dg, "train", shuffle=True)
    val_gen = flow(eval_dg, "val", shuffle=True)
    test_gen = flow(eval_dg, "test", shuffle=False)

    model, base = build_transfer_model(base_class)
    ckpt_path = os.path.join(out_dir, "best_model.keras")

    # ---------------- PHASE 1: frozen base, head only, 20 fixed epochs, NO early stopping ----------------
    model.compile(optimizer=optimizers.Adam(learning_rate=PHASE1_LR),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        train_gen, validation_data=val_gen, epochs=PHASE1_EPOCHS,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
            tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase1.csv")),
        ],
    )

    # ---------------- PHASE 2: unfreeze top 30% of base, LR=1e-5, early stop on val_loss ----------------
    base.trainable = True
    freeze_until = int(len(base.layers) * (1 - UNFREEZE_FRACTION))
    for layer in base.layers[:freeze_until]:
        layer.trainable = False

    model.compile(optimizer=optimizers.Adam(learning_rate=PHASE2_LR),
                   loss="categorical_crossentropy", metrics=["accuracy"])
    model.fit(
        train_gen, validation_data=val_gen, epochs=PHASE2_MAX_EPOCHS,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor="val_loss", mode="min"),
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PHASE2_PATIENCE, restore_best_weights=True),
            tf.keras.callbacks.CSVLogger(os.path.join(out_dir, "training_log_phase2.csv")),
        ],
    )

    model = tf.keras.models.load_model(ckpt_path)
    return evaluate_and_save(model, test_gen, class_names, out_dir, name)

In [ ]:
# =========================================================================
# SHARED EVALUATION - Section 3.5 (accuracy, macro P/R/F1, confusion matrix)
# =========================================================================
def evaluate_and_save(model, test_gen, class_names, out_dir, name):
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes  # ground-truth labels in the (unshuffled) test generator order

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    pd.DataFrame(report).transpose().to_csv(os.path.join(out_dir, "classification_report.csv"))

    cm = confusion_matrix(y_true, y_pred)
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(os.path.join(out_dir, "confusion_matrix.csv"))

    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.close()

    summary_row = {
        "Model": name,
        "Top1_Accuracy": report["accuracy"],
        "Precision_Macro": report["macro avg"]["precision"],
        "Recall_Macro": report["macro avg"]["recall"],
        "F1_Macro": report["macro avg"]["f1-score"],
    }
    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump(summary_row, f, indent=2)

    print(f"{name} done. Top-1 accuracy: {summary_row['Top1_Accuracy']:.4f}")
    return summary_row

In [ ]:
# =========================================================================
# MAIN
# =========================================================================
def main():
    # Pull class names from the train folder (must match the 10 class folder names exactly)
    class_names = sorted(os.listdir(os.path.join(DATA_DIR, "train")))
    print("Classes found:", class_names)

    all_summaries = [train_custom_cnn(class_names)]
    for name, (base_class, preprocess_fn) in TRANSFER_SPECS.items():
        all_summaries.append(train_transfer_model(name, base_class, preprocess_fn, class_names))

    comparison_df = pd.DataFrame(all_summaries).sort_values("Top1_Accuracy", ascending=False)
    comparison_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison_summary.csv"), index=False)
    print("\nFinal comparison:\n", comparison_df)


if __name__ == "__main__":
    main()

Classes found: ['Basketball', 'BenchPress', 'Biking', 'Fencing', 'GolfSwing', 'HorseRiding', 'Kayaking', 'Skiing', 'SoccerJuggling', 'TennisSwing']

Training: Custom_CNN
Found 7202 images belonging to 10 classes.
Found 1607 images belonging to 10 classes.
Found 1195 images belonging to 10 classes.
Epoch 1/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 166s 691ms/step - accuracy: 0.4429 - loss: 1.5754 - val_accuracy: 0.1805 - val_loss: 4.7415
Epoch 2/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 148s 654ms/step - accuracy: 0.6209 - loss: 1.0896 - val_accuracy: 0.2240 - val_loss: 3.2814
Epoch 3/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 145s 640ms/step - accuracy: 0.7009 - loss: 0.8841 - val_accuracy: 0.4611 - val_loss: 1.9761
Epoch 4/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 143s 633ms/step - accuracy: 0.7667 - loss: 0.7028 - val_accuracy: 0.4767 - val_loss: 2.0439
Epoch 5/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 144s 633ms/step - accuracy: 0.7734 - loss: 0.6793 - val_accuracy: 0.4698 - val_loss: 1.6416
Epoch 6/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1